# Hotel Booking Data Analysis

**Course:** Data Processing with Python (HAMK)

**Group members:**
1. Anup Kandel
2. Sandesh Bishokarma
3. Sushant Koirala
4. Tanka Raj Aryal

Each section below lists what needs to be done (from the project description on Moodle) and who is responsible for it.
The data file is in `../data/hotel_bookings_final_project.csv`. Do not change it. Save the exported files to `../outputs/`.

## Context and research questions

**What one row represents:** one hotel booking (completed or cancelled), with details of the stay (dates, nights, guests), how it was booked (market segment, deposit type, customer type) and its price (average daily rate in euros).

**Population and time period:** bookings at a city hotel (Lisbon) and a resort hotel (Algarve), Portugal, with arrival dates from July 2015 to August 2017. This is a 1,000-record sample of the Hotel Booking Demand dataset (António, de Almeida & Nunes, 2019).

**Purpose of the analysis:** the audience is hotel management. We want to help them understand **which bookings are more likely to be cancelled** and **how demand differs between the two hotels**, so they can plan capacity, deposits and pricing.

**Research questions**

1. *(categorical comparison)* Do cancellation rates differ between the City Hotel and the Resort Hotel?
2. *(numerical variable)* Is lead time (days between booking and arrival) associated with cancellation?
3. *(time pattern)* How does booking volume vary by arrival month for each hotel?

**Responsible:** Tanka Raj Aryal. The three research questions are agreed by the whole group at the kickoff (led by Sandesh Bishokarma).

## Imports and display settings

**Responsible:** Tanka Raj Aryal.

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.2f}".format)

# Chart settings from the lesson notebooks
plt.rcParams.update({
    "figure.figsize": (9, 5),
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

# File paths. This file is in notebooks/work/, so the data is two folders up.
# (In the shared notebook Anup changes '../../' to '../'.)
DATA_FILE = "../../data/hotel_bookings_final_project.csv"
CLEAN_FILE = "../../outputs/hotel_bookings_cleaned.csv"
SUMMARY_FILE = "../../outputs/hotel_booking_summary.csv"

## Load and inspect the data

Use Pandas to load the CSV and inspect the dataset: number of rows and columns; column names and data types;
a small preview or random sample; missing values; exact duplicates; useful category counts; numerical summary statistics.
Do not print large amounts of unnecessary output.

**Responsible:** Tanka Raj Aryal.

In [24]:
DATA_FILE = "../data/hotel_bookings_final_project.csv"

hotel_raw = pd.read_csv(DATA_FILE)
print(f"Rows: {hotel_raw.shape[0]}, columns: {hotel_raw.shape[1]}")
hotel_raw.head()

Rows: 1000, columns: 20


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,country,market_segment,deposit_type,customer_type,average_daily_rate,is_repeated_guest,booking_changes,required_car_parking_spaces,total_of_special_requests
0,City Hotel,1,1,2015,September,30,0,2,1,0,0,PRT,Offline TA/TO,Non Refund,Transient,98.10,1,0,0,0
1,Resort Hotel,1,19,2016,March,19,2,4,2,0,0,PRT,Online TA,No Deposit,Transient,70.17,0,0,0,1
2,Resort Hotel,0,9,2017,August,1,0,4,2,0,0,ESP,Online TA,No Deposit,Transient,193.40,0,0,0,1
3,Resort Hotel,0,110,2016,November,11,0,1,2,0,0,BRA,Online TA,No Deposit,Transient,36.24,0,2,1,0
4,City Hotel,0,329,2017,July,27,0,2,2,0,0,GBR,Online TA,No Deposit,Transient,89.10,0,0,0,1


In [25]:
#Rows and Columns
print("Rows and columns:", hotel_raw.shape)
print("Columns:", hotel_raw.columns.tolist())
hotel_raw.info()

Rows and columns: (1000, 20)
Columns: ['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'country', 'market_segment', 'deposit_type', 'customer_type', 'average_daily_rate', 'is_repeated_guest', 'booking_changes', 'required_car_parking_spaces', 'total_of_special_requests']
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   hotel                        1000 non-null   str    
 1   is_canceled                  1000 non-null   int64  
 2   lead_time                    1000 non-null   int64  
 3   arrival_date_year            1000 non-null   int64  
 4   arrival_date_month           1000 non-null   str    
 5   arrival_date_day_of_month    1000 non-null   int64  
 6   stays_in_weekend_nigh

# missing values and exact duplicates

In [26]:
# Missing values and exact duplicates
missing = hotel_raw.isna().sum()
print("Missing values per column (only columns with any):")
print(missing[missing > 0])
print("\nExact duplicate rows:", hotel_raw.duplicated().sum())

# Display the category labels, excluding missing values, in sorted order
print("\nHotel labels:", sorted(hotel_raw["hotel"].dropna().unique()))
print("Deposit labels:", sorted(hotel_raw["deposit_type"].dropna().unique()))
print("Month labels:", hotel_raw["arrival_date_month"].nunique(), "different months")

Missing values per column (only columns with any):
country    2
dtype: int64

Exact duplicate rows: 26

Hotel labels: ['City Hotel', 'Resort Hotel']
Deposit labels: ['No Deposit', 'Non Refund', 'Refundable']
Month labels: 12 different months


In [27]:
# Useful category counts
for col in ["hotel", "is_canceled", "market_segment", "deposit_type", "customer_type"]:
    print(f"--- {col} ---")
    print(hotel_raw[col].value_counts(dropna=False), "\n")

--- hotel ---
hotel
City Hotel      642
Resort Hotel    358
Name: count, dtype: int64 

--- is_canceled ---
is_canceled
0    634
1    366
Name: count, dtype: int64 

--- market_segment ---
market_segment
Online TA        481
Offline TA/TO    209
Groups           163
Direct           100
Corporate         41
Complementary      6
Name: count, dtype: int64 

--- deposit_type ---
deposit_type
No Deposit    881
Non Refund    116
Refundable      3
Name: count, dtype: int64 

--- customer_type ---
customer_type
Transient          742
Transient-Party    226
Contract            28
Group                4
Name: count, dtype: int64 



In [28]:
# Show summary statistics for all numeric columns, rounded to 2 decimal places
hotel_raw.select_dtypes(include="number").describe().round(2)

,is_canceled,lead_time,arrival_date_year,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,average_daily_rate,is_repeated_guest,booking_changes,required_car_parking_spaces,total_of_special_requests
count,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00
mean,0.37,105.06,2016.13,16.13,0.92,2.52,1.86,0.11,0.01,101.21,0.03,0.24,0.06,0.59
std,0.48,106.45,0.70,8.96,0.95,1.80,0.53,0.40,0.09,45.72,0.18,0.67,0.24,0.80
min,0.00,0.00,2015.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.00,19.00,2016.00,8.00,0.00,1.00,2.00,0.00,0.00,68.57,0.00,0.00,0.00,0.00
50%,0.00,69.50,2016.00,16.00,1.00,2.00,2.00,0.00,0.00,94.50,0.00,0.00,0.00,0.00
75%,1.00,157.00,2017.00,24.00,2.00,3.00,2.00,0.00,0.00,125.00,0.00,0.00,0.00,1.00
max,1.00,542.00,2017.00,31.00,8.00,20.00,6.00,3.00,1.00,304.00,1.00,7.00,1.00,5.00


In [29]:
# Which months does each year cover? (important for the time analysis later)
year_coverage = (
    hotel_raw.groupby("arrival_date_year")
    .agg(
        months_covered=("arrival_date_month", "nunique"),
        bookings=("hotel", "count"),
    )
)
year_coverage

,months_covered,bookings
arrival_date_year,,
2015,6,188
2016,12,490
2017,8,322


## Data-quality assessment and cleaning log

At minimum, investigate: missing values; exact duplicates; invalid date combinations; impossible negative values;
bookings with zero total guests; bookings with zero total nights; unusual values in `lead_time`; unusual values in `average_daily_rate`.

Create a cleaning log as a Markdown table:

| Issue checked | Evidence | Decision | Reason |
|---|---|---|---|
| | | | |

Keep the original DataFrame unchanged and create a cleaned copy.

**Responsible:** Tanka Raj Aryal.

In [30]:
# Helper columns used only for the checks
check = hotel_raw.copy()
check["nights_tmp"] = check["stays_in_weekend_nights"] + check["stays_in_week_nights"]
check["guests_tmp"] = check["adults"] + check["children"] + check["babies"]
check["date_tmp"] = pd.to_datetime(
    check["arrival_date_year"].astype(str) + "-" + check["arrival_date_month"] + "-"
    + check["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d", errors="coerce")

def iqr_upper_fence(series):
    """Return the 1.5*IQR upper fence used to flag unusually large values."""
    q1, q3 = series.quantile([0.25, 0.75])
    return q3 + 1.5 * (q3 - q1)

numeric_cols = hotel_raw.select_dtypes("number").columns
checks = {
    "missing values (any column)": hotel_raw.isna().any(axis=1).sum(),
    "exact duplicate rows": hotel_raw.duplicated().sum(),
    "invalid date combinations": check["date_tmp"].isna().sum(),
    "rows with a negative numeric value": (hotel_raw[numeric_cols] < 0).any(axis=1).sum(),
    "zero total guests": (check["guests_tmp"] == 0).sum(),
    "zero total nights": (check["nights_tmp"] == 0).sum(),
    "lead_time above IQR fence": (hotel_raw["lead_time"] > iqr_upper_fence(hotel_raw["lead_time"])).sum(),
    "average_daily_rate above IQR fence": (hotel_raw["average_daily_rate"] > iqr_upper_fence(hotel_raw["average_daily_rate"])).sum(),
    "average_daily_rate == 0": (hotel_raw["average_daily_rate"] == 0).sum(),
}
print("lead_time fence:", round(iqr_upper_fence(hotel_raw["lead_time"]), 1), "days")
print("average_daily_rate fence:", round(iqr_upper_fence(hotel_raw["average_daily_rate"]), 1), "EUR")
pd.Series(checks, name="affected rows").to_frame()

lead_time fence: 364.0 days
average_daily_rate fence: 209.6 EUR


,affected rows
missing values (any column),2
exact duplicate rows,26
invalid date combinations,0
rows with a negative numeric value,0
zero total guests,3
zero total nights,5
lead_time above IQR fence,28
average_daily_rate above IQR fence,25
average_daily_rate == 0,15


In [31]:
# Look at the affected rows before deciding. Example: zero-night and zero-guest bookings
check.loc[(check["nights_tmp"] == 0) | (check["guests_tmp"] == 0),
          ["hotel", "is_canceled", "adults", "children", "babies", "nights_tmp", "average_daily_rate", "market_segment"]]

,hotel,is_canceled,adults,children,babies,nights_tmp,average_daily_rate,market_segment
201,City Hotel,0,0,0,0,0,0.00,Online TA
455,Resort Hotel,0,1,0,0,0,0.00,Groups
461,City Hotel,0,1,0,0,0,0.00,Offline TA/TO
774,City Hotel,0,0,0,0,0,0.00,Offline TA/TO
993,City Hotel,0,0,0,0,0,0.00,Online TA


In [32]:
# Duplicates: which kind of bookings are they?
hotel_raw[hotel_raw.duplicated(keep=False)].sort_values(list(hotel_raw.columns)).head(10)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,country,market_segment,deposit_type,customer_type,average_daily_rate,is_repeated_guest,booking_changes,required_car_parking_spaces,total_of_special_requests
353,City Hotel,0,11,2015,December,5,2,2,2,0,0,PRT,Groups,No Deposit,Transient-Party,104.00,0,0,0,0
638,City Hotel,0,11,2015,December,5,2,2,2,0,0,PRT,Groups,No Deposit,Transient-Party,104.00,0,0,0,0
276,City Hotel,0,55,2015,September,6,2,0,2,0,0,ESP,Groups,No Deposit,Transient-Party,75.00,0,0,0,0
535,City Hotel,0,55,2015,September,6,2,0,2,0,0,ESP,Groups,No Deposit,Transient-Party,75.00,0,0,0,0
116,City Hotel,0,102,2015,October,16,0,2,2,0,0,DEU,Offline TA/TO,No Deposit,Transient-Party,109.00,0,0,0,0
957,City Hotel,0,102,2015,October,16,0,2,2,0,0,DEU,Offline TA/TO,No Deposit,Transient-Party,109.00,0,0,0,0
263,City Hotel,0,134,2016,April,9,1,1,2,0,0,PRT,Groups,No Deposit,Transient-Party,90.00,0,0,0,1
713,City Hotel,0,134,2016,April,9,1,1,2,0,0,PRT,Groups,No Deposit,Transient-Party,90.00,0,0,0,1
486,City Hotel,0,377,2016,October,14,0,2,2,0,0,DEU,Offline TA/TO,No Deposit,Transient-Party,115.00,0,0,0,1
804,City Hotel,0,377,2016,October,14,0,2,2,0,0,DEU,Offline TA/TO,No Deposit,Transient-Party,115.00,0,0,0,1


In [33]:
print("Market segment of duplicated rows:")
print(hotel_raw.loc[hotel_raw.duplicated(keep=False), "market_segment"].value_counts())

print("\nMarket segment of bookings with average_daily_rate == 0:")
print(hotel_raw.loc[hotel_raw["average_daily_rate"] == 0, "market_segment"].value_counts())

Market segment of duplicated rows:
market_segment
Groups           32
Offline TA/TO    19
Name: count, dtype: int64

Market segment of bookings with average_daily_rate == 0:
market_segment
Complementary    6
Online TA        3
Offline TA/TO    3
Groups           2
Direct           1
Name: count, dtype: int64


In [34]:
# The most extreme lead times and daily rates
print("Largest lead times (days):", hotel_raw["lead_time"].nlargest(5).tolist())
print("Largest daily rates (EUR):", hotel_raw["average_daily_rate"].nlargest(5).tolist())

# Rows where the daily rate is present but outside a plausible range of 0-500 EUR
invalid_rates = hotel_raw.loc[
    hotel_raw["average_daily_rate"].notna() &
    ~hotel_raw["average_daily_rate"].between(0, 500)
]
print("Daily rates outside 0-500 EUR:", len(invalid_rates))

Largest lead times (days): [542, 538, 531, 517, 496]
Largest daily rates (EUR): [304.0, 284.2, 283.6, 275.0, 271.0]
Daily rates outside 0-500 EUR: 0


### Cleaning log

| Issue checked | Evidence | Decision | Reason |
|---|---|---|---|
| Missing `country` | 2 rows (both Resort Hotel) | **Fill** with `"Unknown"` | Country is not used in our research questions; keeping the rows keeps their other information. |
| Exact duplicates | 26 rows; all are *Groups* or *Offline TA/TO* bookings | **Keep** | The dataset has no booking ID, and group / tour-operator bookings are often entered as several identical bookings, so these are most likely separate real bookings. |
| Invalid date combinations | 0 rows | No action | Every year–month–day combination converts to a real date. |
| Negative values | 0 rows | No action | No numeric column contains negative values. |
| Zero total guests | 3 rows; all also have 0 nights and a rate of 0 € | **Remove** | A booking with no guests and no nights is not a real stay; these look like placeholder records. |
| Zero total nights | 5 rows (3 are the zero-guest rows above) | **Remove 3, keep 2** | The 2 remaining rows have guests and could be day-use bookings; they stay in the data but have a booking value of 0. |
| Unusual `lead_time` | 28 rows above the 364-day fence; maximum 542 days | **Keep** | Booking 1–1.5 years ahead is rare but possible, and long lead times are central to RQ2. |
| Unusual `average_daily_rate` (incl. 0) | 25 rows above the ~210 € fence (max 304 €); 15 rows at 0 € (6 *Complementary*); none outside 0–500 € | **Keep** | High rates fit peak-season pricing and free stays are real (complimentary/promotional); 3 of the 0 € rows are removed by the zero-guest rule. |

## Transformations and derived variables

Create at least these four derived variables:
- `arrival_date` — proper Pandas date;
- `total_nights` — weekend nights + weekday nights;
- `total_guests` — adults + children + babies;
- `estimated_booking_value` — average daily rate × total nights.

You may create additional useful variables if they support your research questions.

**Responsible:** Sushant Koirala.

## Individual-variable analysis

Analyze at least two categorical variables and two numerical variables.
- Categorical: counts; percentages; valid sample size (n); a suitable chart when useful (bar chart).
- Numerical: valid n; mean; median; standard deviation; minimum and maximum; a suitable visualization such as a histogram or box plot.

**Responsible:** Sushant Koirala.

## Relationships and grouped analysis

Complete at least three grouped or two-variable analyses. Together they must include:
- one categorical-versus-categorical analysis using a crosstab;
- one categorical-versus-numerical analysis using `groupby()` and aggregation; and
- one numerical-versus-numerical or time-based analysis.

For crosstabs, show both counts and clearly labelled percentages. State what the percentage denominator is.

**Responsible:** Sandesh Bishokarma (crosstab for RQ1, groupby for RQ2) and Anup Kandel (time-based analysis for RQ3).

## Final visualizations

At least four finished visualizations (Matplotlib or Pandas plotting): one categorical comparison; one numerical distribution;
one relationship or grouped comparison; one time-based or multi-group figure.

Each needs a descriptive title, clear axis labels, a legend when needed, readable labels, sensible formatting, and the sample size (n).
After each, write a short **What can we observe?** paragraph without unsupported causal claims.

**Responsible:** Anup Kandel puts the final four together. Categorical and grouped comparison: Sandesh Bishokarma. Numerical distribution: Sushant Koirala. Time-based figure: Anup Kandel.

## Conclusions, limitations, and recommendation

- answers to the three research questions;
- three to five evidence-based findings;
- at least two limitations or cautions;
- one recommendation for the hotel-management audience; and
- one useful next question that cannot be answered with the supplied variables.

**Responsible:** Whole group, led by Sandesh Bishokarma.

## Exports and verification

Export at least `hotel_bookings_cleaned.csv` (cleaned dataset) and `hotel_booking_summary.csv` (one useful summary table).
Read the exported files back into Python and verify their shape and key columns.

**Responsible:** Anup Kandel.